# Performance and Optimization Strategy

## Purpose

I use this notebook to evaluate the performance strategy for the Health
Insurance lakehouse.

The project uses relatively small datasets, so I avoid unnecessary manual
optimization operations that would add cost without providing a meaningful
performance benefit.

Instead, I evaluate:

- table size and storage characteristics
- predictive optimization status
- managed-table optimization features
- whether manual partitioning or clustering is justified
- future optimization actions if the workload grows

### Optimization principle

I optimize based on workload evidence rather than applying tuning commands
automatically.

For the current project scale, I prefer Databricks-managed optimization
features over manual partitioning, Z-Ordering, or repeated OPTIMIZE jobs.

In [0]:
%sql
-- inspecting the physical characteristics of the main Silver Claims table.

DESCRIBE DETAIL health_insurance.silver.claims;

In [0]:
%sql
-- inspecting the physical characteristics of the central Gold fact table.

DESCRIBE DETAIL health_insurance.gold.fact_claim;

## Table-size assessment

The current Claims source contains approximately 20,000 records.

At this scale, I do not introduce traditional partitioning because the
additional partition-management overhead would outweigh the likely performance
benefit.

Databricks recommends avoiding manual partitioning for most tables below very
large scale and recommends liquid clustering when explicit data-layout
optimization eventually becomes necessary.

In [0]:
%sql
-- checking whether predictive optimization is enabled for the project catalog.

DESCRIBE CATALOG EXTENDED health_insurance;

## Predictive optimization decision

I first inspect whether predictive optimization is already inherited from the
Databricks account or metastore.

If it is already enabled, I allow Unity Catalog managed tables to inherit the
platform-managed optimization behavior.

If it is disabled, I do not enable it solely for this portfolio workload
because the current datasets are small and additional serverless maintenance
operations would provide limited measurable benefit.

For a larger production deployment, I would evaluate enabling predictive
optimization at the catalog level so Databricks can automatically manage
OPTIMIZE, VACUUM, and ANALYZE operations.

## Data-layout strategy

I do not use static partitions or Z-Ordering for the current project.

The dataset is too small to justify manually tuning physical data layout.

If the Claims fact table grows substantially and analytical queries repeatedly
filter by common dimensions such as claim date, Patient, or Provider, I would
evaluate liquid clustering.

Liquid clustering is preferable to legacy partitioning and Z-Ordering because
clustering keys can evolve with the workload and Databricks can manage the data
layout incrementally.

In [0]:
%sql
-- reviewing Delta Lake operations performed on the Claims Silver table.

DESCRIBE HISTORY health_insurance.silver.claims;

In [0]:
%sql
-- reviewing the operational history of the Gold Claim fact.

DESCRIBE HISTORY health_insurance.gold.fact_claim;

## Final optimization strategy

For the current workload, I deliberately keep the physical optimization design
simple.

### Current decisions

- I use Unity Catalog managed datasets where possible.
- I use serverless compute for pipeline execution.
- I use Lakeflow materialized views for reusable Gold analytical outputs.
- I avoid traditional table partitioning.
- I avoid manual Z-Ordering.
- I avoid recurring manual OPTIMIZE jobs.
- I inspect predictive optimization inheritance before enabling additional
  automated maintenance.
- I retain Delta history for operational inspection.
- I consider liquid clustering only when future workload size and query patterns
  justify it.

### Future production scale

If the project grows substantially, I would evaluate:

1. automatic or explicit liquid clustering for large frequently queried tables
2. predictive optimization at the catalog or schema level
3. query-history analysis to identify common filters and joins
4. pipeline and Job duration metrics
5. file-count and table-size trends
6. clustering-key effectiveness
7. serverless compute and storage cost trends

### Engineering rationale

I do not treat optimization as a checklist of commands.

I choose optimization techniques only when table scale, query patterns, or
operational metrics demonstrate a measurable need.